In [ ]:
import random
from csv import excel
from email import header
import requests
from dotenv import load_dotenv
from pathlib import Path
import sys
import os
import json
from openai import OpenAI

lib_path = str(Path.cwd().parent.parent)
sys.path.append(lib_path)
cwd = Path.cwd().parent.parent.joinpath("env/.env")

if cwd.exists():
    load_dotenv(dotenv_path=cwd, override=True)
else:
    print("No .env file found")
    sys.exit(1)

print(os.environ["APP_NAME"])

In [ ]:
messages = [{"role":"system", "content":"You are snarky assistant named eddie, give you name while providing response"}]
messages += [{"role":"user", "content":"Hi"}]

In [ ]:
openai = OpenAI()
gpt_model = os.getenv("GPT_MODEL")

In [ ]:
messages  += [{"role":"user", "content":"explain:\"north star analogy\""}]
chat = openai.chat.completions.create(messages=messages, model=gpt_model)

In [ ]:
chat.choices[0].message.content

In [ ]:
chat.choices[0].message.content

In [ ]:
""" Making http request to Open AI """
import requests

url = "https://api.openai.com/v1/chat/completions"
header = {"Authorization":f"Bearer {os.getenv('OPENAI_API_KEY')}","content-type":"application/json"}

payload = {
    "model": gpt_model,
    "messages": messages
}

payload2 = {
    "model":os.getenv("GPT_MODEL"),
    "messages": messages
}

response  =  requests.post(url, json=payload, headers=header)
response2  =  requests.post(url, json=payload2, headers=header)

In [ ]:
response.json()

In [ ]:
response2.json()

In [ ]:
""" CONNECTING TO OLLAMA """
import requests
requests.get(f"{os.getenv("OLLAMA_BASE_URL")}/models").json()

In [ ]:
messages +=[{"role":"user", "content":"tell me a joke in hindi"}]

In [ ]:
ollama_ai = OpenAI(base_url=os.getenv("OLLAMA_BASE_URL"), api_key="ollamalocal")
try:
    ollama_response = ollama_ai.chat.completions.create(
        model=os.getenv("LLAMA_MODEL"),
        messages=messages
    )
    ollama_response
except Exception as e:
    print(e)



In [ ]:
print(messages)
ollama_response.choices[0].message.content

In [ ]:
from library.common_bot import (
    extract_links_from_website,
    scrape_and_summarize_website,
    dump_as_json
)
res = extract_links_from_website("https://edwarddonner.com/")

In [ ]:
print(dump_as_json(res))

In [ ]:
links = res["links"]
links

In [ ]:
relative_links_messages = [
    {"role":"system", "content": """You are an assistant with responsibility to extract only related links  of the company and courses.
    group course related link separately
    Your response should be like :

    {
        "links":[
            {
             "type":"<about page>",
             "page_link":"page link"
            },
             {
             "type":"<avatar page>",
             "page_link":"page link"
            }
        ],
        "courses_links:[
         {
             "type":"<course 1>",
             "page_link":"page link"
            },{
             "type":"<course 2>",
             "page_link":"page link"
            }
        ]
    }"""},
    {"role":"user", "content": f"links form website {links}"}
]

# print(relative_links_messages)
try:
    ollama_response = ollama_ai.chat.completions.create(
        model=os.getenv("GEMMA4_MODEL"),
        messages=relative_links_messages,
        response_format={"type":"json_object"}
    )

except Exception as e:
    print(e)

In [ ]:
json.loads(ollama_response.choices[0].message.content)

In [ ]:
"""STREAMING EXAMPLE  """


ollama_stream = OpenAI(base_url=os.getenv("OLLAMA_BASE_URL"), api_key="ollamalocal")

messages_stream = [{"role":"system", "content":"You are snarky assistant named eddie, give you name while providing response"}]
messages_stream += [{"role":"user", "content":"Hi"}]

try:
    messages_stream +=[{"role":"user", "content":"Explain all newton law in detail"}]
    ollama_streamed_response = ollama_stream.chat.completions.create(
        model=os.getenv("GEMMA4_MODEL"),
        messages=messages_stream,
        stream=True
    )
    # ollama_streamed_response
    for c in ollama_streamed_response:
        if response := c.choices[0].delta.content:
            print(response, end="")
except Exception as e:
    print(e)



### For printing results normally -in one go use
    response.choices[0].message.content

### For printing response from streaming use
    c.choices[0].delta.content:
```
for c in ollama_streamed_response:
    if response := c.choices[0].delta.content:
        print(response, end="")
